# Metodo exacto (MILP corregido) vs. algoritmo genetico calibrado

Este notebook:

1. **ECalibrado** - ejecuta el algoritmo genetico calibrado sobre las 3
   instancias y 20 semillas, igual que antes.
2. **EExacto (rejilla de tiempos)** - ejecuta el MILP global sobre
   `small`, `medium` y `large`, con **tres tiempos limite**: $15$~min,
   $30$~min y $1$~h, para poder documentar si el método
   exacto llega a una solucion fiable y como evoluciona con más tiempo.
3. Una **tabla de progresion** por instancia y tiempo (UB, LB, gap,
   `optimo_certificado`, si hubo o no solucion factible), y la comparativa
   final con el LCOH del genetico.

---
## Celda 1 - Montar Drive y localizar `EXPERIMENTOS/`

In [4]:
# =====================================================================
# CELDA 1 - SETUP: montar Drive, localizar EXPERIMENTOS/ e importar
# =====================================================================
import os, sys, time
import pandas as pd
import pulp

DRIVE_OK = False
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_OK = True
except Exception as _e:
    print("[INFO] No se pudo montar Drive (fuera de Colab?):", _e)

DIR_NOTEBOOK = None
raiz_busqueda = "/content/drive" if os.path.isdir("/content/drive") else os.getcwd()
candidatos = []
for raiz, _dirs, ficheros in os.walk(raiz_busqueda):
    if "_rutas.py" in ficheros:
        candidatos.append(raiz)
candidatos.sort()
if candidatos:
    DIR_NOTEBOOK = candidatos[0]
    print("Carpeta EXPERIMENTOS encontrada:", DIR_NOTEBOOK)
    DIR_TFM_DETECTADO = os.path.dirname(DIR_NOTEBOOK)
    print("Carpeta TFM detectada         :", DIR_TFM_DETECTADO)
else:
    raise FileNotFoundError(
        "No se encontro ninguna carpeta con '_rutas.py' dentro de " + raiz_busqueda + "."
    )

if DIR_NOTEBOOK not in sys.path:
    sys.path.insert(0, DIR_NOTEBOOK)

import _rutas
import calibracion
import config_experimentos as C
import exportar
import exportar_calibracion
import graficas
import instancias as ins
import milp_global_exacto as milp

print("DIR_RAIZ       :", _rutas.DIR_RAIZ)
print("DIR_RESULTADOS :", _rutas.DIR_RESULTADOS)
print("Instancias     :", C.INSTANCIAS)






Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta EXPERIMENTOS encontrada: /content/drive/MyDrive/TFM/EXPERIMENTOS
Carpeta TFM detectada         : /content/drive/MyDrive/TFM
DIR_RAIZ       : /content/drive/MyDrive/TFM/CODIGO_FUENTE
DIR_RESULTADOS : /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS
Instancias     : ['small', 'medium', 'large']


---
## Celda 2 - Utilidad: resumen de una fase

In [5]:
# =====================================================================
# CELDA 2 - UTILIDADES
# =====================================================================
def resumen_fase(exp_id, r):
    res = r["resumen"]
    runs = r["runs"]
    cols_rejilla = r["cols_rejilla"]
    print("=" * 78)
    print(f"RESUMEN DE {exp_id}")
    print("=" * 78)
    t_total = pd.to_numeric(runs["tiempo_s"], errors="coerce").fillna(0.0).sum()
    print(f"Tiempo total real: {t_total:.1f} s ({t_total/60:.1f} min)\n")
    ganadoras = {}
    for inst, sub in res.groupby("instancia", sort=False):
        g = sub[sub["es_mejor_en_instancia"] == True]
        if g.empty:
            continue
        fila = g.iloc[0]
        ganadoras[inst] = fila["id_config"]
        print(f"  [{inst:<8}] LCOH medio = {fila['lcoh_media']:.4f}  "
              f"std = {fila['lcoh_std']:.4f}  factibilidad = {fila['tasa_factibilidad']:.2f}")
    print("=" * 78)
    return ganadoras


---
## FASE A - `ECalibrado`: algoritmo genetico con los hiperparametros calibrados


In [6]:
# =====================================================================
# ECalibrado - VALORES CALIBRADOS (edita con tus valores finales)
# =====================================================================
VALORES_CALIBRADOS = {
    "prob_mutacion":   0.2,
    "prob_cruce":      0.6,
    "tam_poblacion":   150,
    "n_generaciones":  200,
    "k_torneo":        2,
    "tipo_init":       "B",
    "frac_semilla":    0.5,
    "m_ref":           "min_efi",
}

faltan = [k for k, v in VALORES_CALIBRADOS.items() if v is None]
if faltan:
    raise ValueError(f"Faltan valores calibrados sin fijar: {faltan}")

print("Valores calibrados que se van a usar en ECalibrado:")
for k, v in VALORES_CALIBRADOS.items():
    print(f"  {k:<16} {v}")
for k, v in VALORES_CALIBRADOS.items():
    C.CALIBRADO[k] = v


Valores calibrados que se van a usar en ECalibrado:
  prob_mutacion    0.2
  prob_cruce       0.6
  tam_poblacion    150
  n_generaciones   200
  k_torneo         2
  tipo_init        B
  frac_semilla     0.5
  m_ref            min_efi


In [7]:
# =====================================================================
# ECalibrado - EJECUCION
# =====================================================================
C.EXPERIMENTOS["ECalibrado"] = {
    "nombre": "Ejecucion final con hiperparametros calibrados",
    "rejilla": {"tipo_init": [VALORES_CALIBRADOS["tipo_init"]]},
}

r_ECAL = calibracion.ejecutar_calibracion(
    exp_id="ECalibrado", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="ECalibrado", res=r_ECAL["resumen"], runs=r_ECAL["runs"],
    conv=pd.read_csv(r_ECAL["ruta_convergencia"]) if os.path.isfile(r_ECAL["ruta_convergencia"]) else None,
    cols_rejilla=r_ECAL["cols_rejilla"], nombre_exp=r_ECAL["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="ECalibrado", runs=r_ECAL["runs"], res=r_ECAL["resumen"],
    ranking=r_ECAL.get("ranking"), nombre_exp=r_ECAL["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_ECAL["resumen"]["instancia"].tolist())):
    sub = r_ECAL["resumen"][r_ECAL["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_ECAL["runs"].loc[r_ECAL["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_ECAL["runs"].loc[r_ECAL["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="ECalibrado", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_ECAL["cols_rejilla"], nombre_exp=r_ECAL["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )

ganadoras_ECAL = resumen_fase("ECalibrado", r_ECAL)
LCOH_GA = r_ECAL["resumen"].set_index("instancia")["lcoh_media"].to_dict()
print("\\nLCOH medio del algoritmo genetico calibrado, por instancia:")
for inst, v in LCOH_GA.items():
    print(f"  {inst:<8} {v:.4f} EUR/kg")


ECalibrado  |  Ejecucion final con hiperparametros calibrados
  Instancias      : small, medium, large
  Configuraciones : 1  (init=B)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 60
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=2, init=B

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
    [pool] recuperado de cache: pool_small_nogood_8_0p15_min_efi_12345_0p1_5p0.pkl
   init=B                       fact=20/20  LCOH medio=4.6046          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
    [pool] recuperado de cache: pool_medium_nogood_8_0p15_min_efi_12345_0p1_5p0.pkl
   init=B                       fact=20/20  LCOH medio=4.3720          [ 66.7%]

[large]  |P|=12 |J|=30 |K|=55 HTotal=53000 kg/dia
    [pool] recuperado de cache: pool_large_nogood_8_0p15_min_efi_12345_0p1_5p0.pkl
   init=B                       fact=20/20  LCOH medio=4.4452          [100.0%]

Co

---
## FASE B - `EExacto`: MILP global, con rejilla de tiempos limite

Se prueban **tres tiempos limite** por instancia: $900$ s ($15$ min),
$1\,800$ s ($30$ min) y $3\,600$~s ($1$h). Cada combinacion
(instancia, timeout) se ejecuta y se guarda de forma independiente en
`RESULTADOS/EExacto/EExacto_resultados.csv` (se actualiza esa fila si ya
existia, sin perder las demas).



In [8]:
# =====================================================================
# EExacto - PARAMETROS Y FUNCIONES COMUNES
# =====================================================================
REJILLA_TIMEOUTS_S = {
    "15min": 900.0,
    "30min": 1800.0,
    "1h":    3600.0,
}
GAP_ACEPTADO: float = 0.0
MOSTRAR_LOG_CBC: bool = False

DIR_EEXACTO = os.path.join(_rutas.DIR_RESULTADOS, "EExacto")
os.makedirs(DIR_EEXACTO, exist_ok=True)
RUTA_EEXACTO_CSV = os.path.join(DIR_EEXACTO, "EExacto_resultados.csv")


def resolver_solo_objetivo(inst, timeout, gap=0.0, msg=False):
    """Construye y resuelve el MILP global extrayendo solo el valor de
    la funcion objetivo (UB) y, si esta disponible, el best bound de CBC (LB).

    Solo se acepta el objetivo si las variables binarias con valor asignado
    son un INCUMBENTE ENTERO real (milp._es_incumbente_entero). Si CBC para
    por timeout sin haber encontrado ningun incumbente entero, PuLP puede
    dejar en las variables los valores de la relajacion LP del ultimo nodo
    explorado (fracciones); esa relajacion es una cota inferior teorica del
    optimo, NUNCA una solucion factible del problema real.
    """
    t0 = time.time()
    prob, variables = milp._construir(inst)

    solver = pulp.PULP_CBC_CMD(msg=1 if msg else 0, timeLimit=int(timeout), gapRel=gap)
    prob.solve(solver)

    tiempo_real = time.time() - t0
    estado = pulp.LpStatus.get(prob.status, str(prob.status))

    objetivo = None
    hay_valores = any(v.value() is not None for v in variables["y"].values())
    if hay_valores and milp._es_incumbente_entero(variables):
        try:
            objetivo = pulp.value(prob.objective)
        except Exception:
            objetivo = None

    cota_inferior = None
    for attr in ("bestBound", "best_bound"):
        val = getattr(prob, attr, None)
        if val is not None:
            try:
                cota_inferior = float(val)
                break
            except (TypeError, ValueError):
                pass

    gap_final = None
    if objetivo is not None and cota_inferior is not None and objetivo != 0:
        gap_final = abs(objetivo - cota_inferior) / abs(objetivo)

    return {
        "estado": estado,
        "objetivo": objetivo,
        "cota_inferior": cota_inferior,
        "gap_final": gap_final,
        "optimo_certificado": (estado == "Optimal"),
        "tiempo_s_real": tiempo_real,
        "n_variables": len(prob.variables()),
        "n_restricciones": len(prob.constraints),
    }


def ejecutar_combo_exacto(inst_id, etiqueta_timeout, timeout_s,
                            gap=GAP_ACEPTADO, msg=MOSTRAR_LOG_CBC):
    """Resuelve UNA (instancia, timeout) y actualiza su fila en el CSV
    compartido, identificada por (instancia, etiqueta_timeout), sin afectar
    a las demas filas ya guardadas."""
    etiq = ins.etiqueta(inst_id)
    inst = ins.cargar(inst_id)

    print("=" * 78)
    print(f"[{etiq} | {etiqueta_timeout}]  |P|={len(inst.P)} |J|={len(inst.J)} "
          f"|K|={len(inst.K)} HTotal={inst.HTotal:.0f} kg/dia   "
          f"->   timeout={timeout_s:.0f} s")
    print("=" * 78)

    r = resolver_solo_objetivo(inst, timeout=timeout_s, gap=gap, msg=msg)

    if r["objetivo"] is not None:
        certeza = "OPTIMO CERTIFICADO" if r["optimo_certificado"] else "cota superior (sin certificar)"
        print(f"[{etiq} | {etiqueta_timeout}] UB = {r['objetivo']:.2f}  [{certeza}]")
    else:
        print(f"[{etiq} | {etiqueta_timeout}] SIN incumbente ENTERO factible en "
              f"{timeout_s:.0f}s -> no hay cota superior.")
    print(f"[{etiq} | {etiqueta_timeout}] LB (best bound) = {r['cota_inferior']}")
    print(f"[{etiq} | {etiqueta_timeout}] tiempo real: {r['tiempo_s_real']:.1f} s "
          f"(timeout pedido: {timeout_s:.0f} s)\\n")

    lcoh_ub = (r["objetivo"] / inst.HTotal) if r["objetivo"] is not None else None
    lcoh_lb = (r["cota_inferior"] / inst.HTotal) if r["cota_inferior"] is not None else None

    fila = {
        "instancia": etiq,
        "timeout_etiqueta": etiqueta_timeout,
        "P": len(inst.P), "J": len(inst.J), "K": len(inst.K), "HTotal": inst.HTotal,
        "timeout_s": timeout_s,
        "gap_aceptado": gap,
        "estado_solver": r["estado"],
        "optimo_certificado": r["optimo_certificado"],
        "tiene_solucion_factible": r["objetivo"] is not None,
        "coste_total_UB": r["objetivo"],
        "coste_total_LB": r["cota_inferior"],
        "lcoh_UB": lcoh_ub,
        "lcoh_LB": lcoh_lb,
        "gap_final": r["gap_final"],
        "tiempo_s_real": r["tiempo_s_real"],
        "n_variables": r["n_variables"],
        "n_restricciones": r["n_restricciones"],
    }

    # Actualiza el CSV: reemplaza la fila con la misma (instancia, timeout_etiqueta)
    # si ya existia, o la anade si es nueva. Asi cada combinacion es independiente.
    if os.path.isfile(RUTA_EEXACTO_CSV):
        df_actual = pd.read_csv(RUTA_EEXACTO_CSV)
        mask = ~((df_actual["instancia"] == etiq) &
                 (df_actual["timeout_etiqueta"] == etiqueta_timeout))
        df_actual = df_actual[mask]
        df_actual = pd.concat([df_actual, pd.DataFrame([fila])], ignore_index=True)
    else:
        df_actual = pd.DataFrame([fila])
    df_actual.to_csv(RUTA_EEXACTO_CSV, index=False)
    print(f"Guardado -> {RUTA_EEXACTO_CSV}\\n")

    return fila

print("Rejilla de timeouts configurada:")
for etq, s in REJILLA_TIMEOUTS_S.items():
    print(f"  {etq:<6} = {s:.0f} s")


Rejilla de timeouts configurada:
  15min  = 900 s
  30min  = 1800 s
  1h     = 3600 s


### Pruebas Realizadas

Cada prueba tarda hasta
el tiempo limite indicado (puede terminar antes si CBC certifica el
optimo).

In [11]:
# =====================================================================
# EExacto - SMALL @ 15min
# =====================================================================
fila_small_15min = ejecutar_combo_exacto(
    "small", "15min", REJILLA_TIMEOUTS_S["15min"],
)
fila_small_15min


[small | 15min]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia   ->   timeout=900 s
[small | 15min] SIN incumbente ENTERO factible en 900s -> no hay cota superior.
[small | 15min] LB (best bound) = None
[small | 15min] tiempo real: 897.9 s (timeout pedido: 900 s)\n
Guardado -> /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS/EExacto/EExacto_resultados.csv\n


{'instancia': 'small',
 'timeout_etiqueta': '15min',
 'P': 4,
 'J': 8,
 'K': 16,
 'HTotal': 22400.0,
 'timeout_s': 900.0,
 'gap_aceptado': 0.0,
 'estado_solver': 'Not Solved',
 'optimo_certificado': False,
 'tiene_solucion_factible': False,
 'coste_total_UB': None,
 'coste_total_LB': None,
 'lcoh_UB': None,
 'lcoh_LB': None,
 'gap_final': None,
 'tiempo_s_real': 897.939909696579,
 'n_variables': 10596,
 'n_restricciones': 8684}

In [10]:
# =====================================================================
# EExacto - SMALL @ 30min
# =====================================================================
fila_small_30min = ejecutar_combo_exacto(
    "small", "30min", REJILLA_TIMEOUTS_S["30min"],
)
fila_small_30min


[small | 30min]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia   ->   timeout=1800 s
[small | 30min] SIN incumbente ENTERO factible en 1800s -> no hay cota superior.
[small | 30min] LB (best bound) = None
[small | 30min] tiempo real: 1796.8 s (timeout pedido: 1800 s)\n
Guardado -> /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS/EExacto/EExacto_resultados.csv\n


{'instancia': 'small',
 'timeout_etiqueta': '30min',
 'P': 4,
 'J': 8,
 'K': 16,
 'HTotal': 22400.0,
 'timeout_s': 1800.0,
 'gap_aceptado': 0.0,
 'estado_solver': 'Not Solved',
 'optimo_certificado': False,
 'tiene_solucion_factible': False,
 'coste_total_UB': None,
 'coste_total_LB': None,
 'lcoh_UB': None,
 'lcoh_LB': None,
 'gap_final': None,
 'tiempo_s_real': 1796.777123451233,
 'n_variables': 10596,
 'n_restricciones': 8684}

In [9]:
# =====================================================================
# EExacto - SMALL @ 1h
# =====================================================================
fila_small_1h = ejecutar_combo_exacto(
    "small", "1h", REJILLA_TIMEOUTS_S["1h"],
)
fila_small_1h


[small | 1h]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia   ->   timeout=3600 s
[small | 1h] SIN incumbente ENTERO factible en 3600s -> no hay cota superior.
[small | 1h] LB (best bound) = None
[small | 1h] tiempo real: 3597.7 s (timeout pedido: 3600 s)\n
Guardado -> /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS/EExacto/EExacto_resultados.csv\n


{'instancia': 'small',
 'timeout_etiqueta': '1h',
 'P': 4,
 'J': 8,
 'K': 16,
 'HTotal': 22400.0,
 'timeout_s': 3600.0,
 'gap_aceptado': 0.0,
 'estado_solver': 'Not Solved',
 'optimo_certificado': False,
 'tiene_solucion_factible': False,
 'coste_total_UB': None,
 'coste_total_LB': None,
 'lcoh_UB': None,
 'lcoh_LB': None,
 'gap_final': None,
 'tiempo_s_real': 3597.7226872444153,
 'n_variables': 10596,
 'n_restricciones': 8684}

In [12]:
# =====================================================================
# EExacto - MEDIUM @ 15min
# =====================================================================
fila_medium_15min = ejecutar_combo_exacto(
    "medium", "15min", REJILLA_TIMEOUTS_S["15min"],
)
fila_medium_15min


[medium | 15min]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia   ->   timeout=900 s
[medium | 15min] SIN incumbente ENTERO factible en 900s -> no hay cota superior.
[medium | 15min] LB (best bound) = None
[medium | 15min] tiempo real: 974.4 s (timeout pedido: 900 s)\n
Guardado -> /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS/EExacto/EExacto_resultados.csv\n


{'instancia': 'medium',
 'timeout_etiqueta': '15min',
 'P': 8,
 'J': 18,
 'K': 34,
 'HTotal': 38600.0,
 'timeout_s': 900.0,
 'gap_aceptado': 0.0,
 'estado_solver': 'Not Solved',
 'optimo_certificado': False,
 'tiene_solucion_factible': False,
 'coste_total_UB': None,
 'coste_total_LB': None,
 'lcoh_UB': None,
 'lcoh_LB': None,
 'gap_final': None,
 'tiempo_s_real': 974.3609726428986,
 'n_variables': 108932,
 'n_restricciones': 84322}

In [13]:
# =====================================================================
# EExacto - MEDIUM @ 30min
# =====================================================================
fila_medium_30min = ejecutar_combo_exacto(
    "medium", "30min", REJILLA_TIMEOUTS_S["30min"],
)
fila_medium_30min


[medium | 30min]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia   ->   timeout=1800 s
[medium | 30min] SIN incumbente ENTERO factible en 1800s -> no hay cota superior.
[medium | 30min] LB (best bound) = None
[medium | 30min] tiempo real: 1768.7 s (timeout pedido: 1800 s)\n
Guardado -> /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS/EExacto/EExacto_resultados.csv\n


{'instancia': 'medium',
 'timeout_etiqueta': '30min',
 'P': 8,
 'J': 18,
 'K': 34,
 'HTotal': 38600.0,
 'timeout_s': 1800.0,
 'gap_aceptado': 0.0,
 'estado_solver': 'Not Solved',
 'optimo_certificado': False,
 'tiene_solucion_factible': False,
 'coste_total_UB': None,
 'coste_total_LB': None,
 'lcoh_UB': None,
 'lcoh_LB': None,
 'gap_final': None,
 'tiempo_s_real': 1768.659679889679,
 'n_variables': 108932,
 'n_restricciones': 84322}

In [14]:
# =====================================================================
# EExacto - MEDIUM @ 1h
# =====================================================================
fila_medium_1h = ejecutar_combo_exacto(
    "medium", "1h", REJILLA_TIMEOUTS_S["1h"],
)
fila_medium_1h


[medium | 1h]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia   ->   timeout=3600 s
[medium | 1h] SIN incumbente ENTERO factible en 3600s -> no hay cota superior.
[medium | 1h] LB (best bound) = None
[medium | 1h] tiempo real: 5319.7 s (timeout pedido: 3600 s)\n
Guardado -> /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS/EExacto/EExacto_resultados.csv\n


{'instancia': 'medium',
 'timeout_etiqueta': '1h',
 'P': 8,
 'J': 18,
 'K': 34,
 'HTotal': 38600.0,
 'timeout_s': 3600.0,
 'gap_aceptado': 0.0,
 'estado_solver': 'Not Solved',
 'optimo_certificado': False,
 'tiene_solucion_factible': False,
 'coste_total_UB': None,
 'coste_total_LB': None,
 'lcoh_UB': None,
 'lcoh_LB': None,
 'gap_final': None,
 'tiempo_s_real': 5319.690043210983,
 'n_variables': 108932,
 'n_restricciones': 84322}

In [ ]:
# =====================================================================
# EExacto - LARGE @ 15min
# =====================================================================
fila_large_15min = ejecutar_combo_exacto(
    "large", "15min", REJILLA_TIMEOUTS_S["15min"],
)
fila_large_15min


---
## Tabla de progresion: como evoluciona cada instancia con mas tiempo

Une, para cada instancia, los resultados de las combinaciones que
ejecutadas (no falla si falta alguna). Permite ver, por ejemplo, si el UB
mejora al pasar de 15 a 30 minutos, o si se mantiene estancado.

In [15]:
# =====================================================================
# TABLA DE PROGRESION por instancia y tiempo limite
# =====================================================================
df_exacto = pd.read_csv(RUTA_EEXACTO_CSV)

orden_timeout = {"15min": 900.0, "30min": 1800.0, "1h": 3600.0}
df_exacto["_orden"] = df_exacto["timeout_etiqueta"].map(orden_timeout)
df_exacto = df_exacto.sort_values(["instancia", "_orden"]).drop(columns="_orden")

cols_mostrar = ["instancia", "timeout_etiqueta", "estado_solver",
                "optimo_certificado", "tiene_solucion_factible",
                "coste_total_UB", "lcoh_UB", "coste_total_LB", "lcoh_LB",
                "gap_final", "tiempo_s_real"]
print(df_exacto[cols_mostrar].to_string(index=False))
df_exacto[cols_mostrar]


instancia timeout_etiqueta estado_solver  optimo_certificado  tiene_solucion_factible  coste_total_UB  lcoh_UB  coste_total_LB  lcoh_LB  gap_final  tiempo_s_real
   medium            15min    Not Solved               False                    False             NaN      NaN             NaN      NaN        NaN     974.360973
   medium            30min    Not Solved               False                    False             NaN      NaN             NaN      NaN        NaN    1768.659680
   medium               1h    Not Solved               False                    False             NaN      NaN             NaN      NaN        NaN    5319.690043
    small            15min    Not Solved               False                    False             NaN      NaN             NaN      NaN        NaN     897.939910
    small            30min    Not Solved               False                    False             NaN      NaN             NaN      NaN        NaN    1796.777123
    small               1h  

,instancia,timeout_etiqueta,estado_solver,optimo_certificado,tiene_solucion_factible,coste_total_UB,lcoh_UB,coste_total_LB,lcoh_LB,gap_final,tiempo_s_real
3,medium,15min,Not Solved,False,False,NaN,NaN,NaN,NaN,NaN,974.360973
4,medium,30min,Not Solved,False,False,NaN,NaN,NaN,NaN,NaN,1768.659680
5,medium,1h,Not Solved,False,False,NaN,NaN,NaN,NaN,NaN,5319.690043
2,small,15min,Not Solved,False,False,NaN,NaN,NaN,NaN,NaN,897.939910
1,small,30min,Not Solved,False,False,NaN,NaN,NaN,NaN,NaN,1796.777123
0,small,1h,Not Solved,False,False,NaN,NaN,NaN,NaN,NaN,3597.722687
